# 01 — SQL Injection (классический)

> **`vuln_class`:** `SQL_INJ_CLASSIC` · **Риск:** 10/10 · **CWE-89** · **CAPEC-66**

## Что мы покажем

1. Сделаем мок-БД пользователей с паролями.
2. Напишем **уязвимую** функцию логина — где ввод склеивается с SQL через `+`.
3. Запустим **атаку** и увидим, как утекают данные.
4. Покажем, как наш **аудитор Phase 1** ловит проблему.
5. Напишем **безопасную** версию с параметризацией.
6. Повторим ту же атаку — она уходит в пустоту.


## 🧒 Аналогия для ребёнка

У тебя есть копилка с замком и в ней лежат записки с именами друзей.
Когда друг приходит, ты ищешь его записку.

- **Плохой код** — это когда ты приклеиваешь имя друга прямо на дверь
  копилки и читаешь, что получилось. Если друг скажет
  `«Вася ИЛИ ВСЕ»`, ты прочитаешь именно `«Вася ИЛИ ВСЕ»` —
  и достанешь **ВСЕ** записки сразу.
- **Хороший код** — у тебя есть отдельная картотека с именами.
  Сначала ищешь имя по картотеке, потом достаёшь нужную запись.
  Никакие хитрые слова не работают, потому что ты не «приклеиваешь»
  их к двери — ты ищешь по точному имени.

SQL Injection — когда атакующий пишет в поле ввода логина не своё имя,
а **дополнительный кусок SQL**, который БД честно выполняет.


## 1. Setup — мок-БД пользователей

Создаём in-memory SQLite. Никакого реального Postgres не нужно — нам
важна **семантика SQL**, а она в SQLite такая же на нашем уровне.


In [ ]:
"""
@brief Подготовка окружения и mock-БД через in-memory SQLite.
@details
    Никаких внешних зависимостей кроме stdlib + sqlite3 (есть в Colab из коробки).
    SQLite используем как «упрощённую модель PostgreSQL» — он умеет
    почти весь стандартный SQL, что достаточно для демонстраций уязвимостей.
@note
    Реальная система работает на PostgreSQL (см. ADR-0001),
    использует pglast для AST-парсинга. Здесь, для наглядности,
    эмулируем аудитор через `re` (регулярки) и простой pattern matching.
"""
import sqlite3
import re
import time
from textwrap import dedent


def section(title):
    """@brief Печатает заголовок секции."""
    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)


def show_result(rows, max_rows=10):
    """@brief Печатает результаты запроса в виде таблицы."""
    if not rows:
        print("  (нет строк)")
        return
    for i, r in enumerate(rows[:max_rows]):
        print(f"  {i + 1:>3}. {r}")
    if len(rows) > max_rows:
        print(f"  ... ещё {len(rows) - max_rows} строк")


def print_finding(f):
    """@brief Красиво печатает Finding от нашего аудитора."""
    print(f"  ⚠️  {f['rule_id']}")
    print(f"      vuln_class:  {f['vuln_class']}")
    print(f"      severity:    {f['severity']}")
    print(f"      risk_score:  {f['risk_score']}/10")
    print(f"      message:     {f['message']}")
    if f.get("evidence_refs"):
        print(f"      ссылки:      {', '.join(f['evidence_refs'])}")


##
# @brief Создаёт мок-БД пользователей с 5 тестовыми записями.
# @return sqlite3.Connection с готовой таблицей `users`.
# @note
#   В реальной системе таблица берётся из data_model_sql/data_model.sql
#   и развёрнута в Docker-контейнере с миграциями Alembic.
def setup_users_db():
    conn = sqlite3.connect(":memory:")
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE users (
            id            INTEGER PRIMARY KEY,
            login         TEXT NOT NULL,
            full_name     TEXT,
            password_hash TEXT,   -- В проде здесь HASH, тут plain для наглядности
            role          TEXT
        )
    """)
    cur.executemany(
        "INSERT INTO users (login, full_name, password_hash, role) VALUES (?, ?, ?, ?)",
        [
            ("admin",   "Анна Админовна", "hash_5f8a_admin",  "admin"),
            ("ivanov",  "Иван Иванов",    "hash_2c4b_iv",     "user"),
            ("petrova", "Мария Петрова",  "hash_9d3e_pet",    "user"),
            ("smith",   "John Smith",     "hash_a1b2_smith",  "user"),
            ("test",    "Тест Тестов",    "hash_dead_test",   "user"),
        ],
    )
    conn.commit()
    return conn


conn = setup_users_db()
section("Что лежит в БД (всего у нас 5 пользователей)")
show_result(conn.execute("SELECT id, login, full_name, role FROM users").fetchall())


## 2. Уязвимая функция логина

Самый типичный антипаттерн — склейка ввода через f-string.
Видишь `f"... = '{login}'"`? Это означает «что бы пользователь
ни прислал, оно вставится в SQL ДОСЛОВНО».


In [ ]:
##
# @brief УЯЗВИМАЯ функция: ищет пользователя по логину через конкатенацию.
# @param login  Логин пользователя, пришёл от клиента.
# @return       Список найденных пользователей (login, full_name, role).
# @warning      SQL Injection через `||` и f-string!
# @details
#   Любой ввод подставляется в текст запроса дословно.
#   Если пользователь пришлёт `admin' OR '1'='1`, получится:
#       SELECT * FROM users WHERE login = 'admin' OR '1'='1'
#   и БД вернёт все строки.
def get_user_BAD(conn, login: str):
    # ⚠️ ЗДЕСЬ И ЕСТЬ УЯЗВИМОСТЬ — f-string + конкатенация
    sql = f"SELECT id, login, full_name, role FROM users WHERE login = '{login}'"
    print(f"  Сформированный SQL: {sql}")
    return conn.execute(sql).fetchall()


section("Нормальный сценарий — пользователь честно ввёл свой логин")
rows = get_user_BAD(conn, "ivanov")
print("  Результат:")
show_result(rows)


## 3. Атака — пользователь вводит SQL вместо логина

Атакующий вместо логина `ivanov` присылает специальную строку,
которая «закрывает» исходную кавычку и добавляет условие
`OR '1'='1'` — всегда истинное.


In [ ]:
section("АТАКА: payload вместо логина")
payload = "x' OR '1'='1"
print(f"  Атакующий прислал login = {payload!r}")
rows = get_user_BAD(conn, payload)
print(f"\n  💀 Получено {len(rows)} строк (а должна быть 0 или 1):")
show_result(rows)


section("Усиленная атака: с UNION SELECT — попробуем достать пароли")
# Атакующий уже знает структуру (либо через первую атаку, либо по логам)
payload2 = "x' UNION SELECT id, login, password_hash, role FROM users --"
print(f"  payload = {payload2!r}")
rows = get_user_BAD(conn, payload2)
print(f"\n  💀 Получено {len(rows)} строк, причём с password_hash:")
show_result(rows)


## 4. Что только что произошло

Мы передали в функцию **не логин, а часть SQL**. БД честно выполнила
запрос целиком. В итоге атакующий:

1. Получил **всех пользователей**, не зная ни одного логина.
2. Через `UNION SELECT` достал **password_hash** для всех учёток.

В реальной системе следующий шаг — попытка крекнуть hash и взять
учётку администратора. Защита провалена.


## 5. Аудитор Phase 1 — детектит проблему

В реальной системе наш Phase 1 правило `R011-injection-marker`
использует **`pglast`** (AST-парсинг). Здесь — упрощённый regex-чек
для наглядности.

Идея: ищем в коде функции **маркеры конкатенации** ввода в SQL —
f-string с `{var}` внутри SQL-строки, `+ var +`, `.format(...)`,
`% var`.


In [ ]:
import inspect

INJECTION_MARKERS = [
    (r"f[\"\'].*=\s*[\"\']\{", "f-string подставляет переменную внутрь кавычек"),
    (r"\.execute\(\s*f[\"\']", ".execute(f'...{var}...') — конкатенация в SQL"),
    (r"\.execute\(\s*[\"\'].*[\"\']\s*\+", ".execute('...' + var) — склейка через +"),
    (r"\.execute\(\s*[\"\'].*[\"\'].*%\s*\w", "%-форматирование внутри execute"),
    (r"\.execute\(\s*[\"\'].*\{[^?]*\}.*[\"\']", "format(...) подстановка в SQL"),
]


##
# @brief Упрощённый аудитор Phase 1 (правило R011).
# @details
#   В реальной системе обходим AST через pglast. Здесь — regex поверх
#   исходного кода функции. Берём src через inspect.getsource,
#   а если он недоступен (например, при exec без файла) — принимаем
#   текст функции аргументом.
# @param func_or_src  Python-функция ИЛИ её исходник как строка.
# @return             Список findings (или []).
def audit_phase1_classic_injection(func_or_src):
    if isinstance(func_or_src, str):
        src = func_or_src
    else:
        try:
            src = inspect.getsource(func_or_src)
        except (OSError, TypeError):
            src = repr(func_or_src)  # последний fallback
    findings = []
    for pattern, message in INJECTION_MARKERS:
        if re.search(pattern, src):
            findings.append({
                "rule_id":       "R011-injection-marker",
                "vuln_class":    "SQL_INJ_CLASSIC",
                "severity":      "high",
                "risk_score":    10,
                "message":       message,
                "evidence_refs": ["CWE-89", "CAPEC-66", "OWASP-SQLi-CS"],
            })
            break  # одного маркера достаточно
    return findings


section("Аудитор проверяет get_user_BAD")
for f in audit_phase1_classic_injection(get_user_BAD):
    print_finding(f)


## 6. Безопасная функция — параметризация

Та же логика, но через параметризацию. Драйвер `sqlite3` (как и
`psycopg` для Postgres) сам подставит значение в подготовленный
план — никакой текстовой склейки.


In [ ]:
##
# @brief Безопасная функция: ищет пользователя по логину через параметризацию.
# @param login  Логин (любой текст).
# @return       Список (id, login, full_name, role).
# @note
#   `?` — placeholder для параметра. Драйвер передаёт значение отдельно от
#   текста запроса. В psycopg для PostgreSQL placeholder — `%s`.
def get_user_GOOD(conn, login: str):
    sql = "SELECT id, login, full_name, role FROM users WHERE login = ?"
    print(f"  SQL (без интерполяции): {sql}")
    print(f"  Параметр (отдельно):    login = {login!r}")
    return conn.execute(sql, (login,)).fetchall()


section("Аудитор проверяет get_user_GOOD")
findings = audit_phase1_classic_injection(get_user_GOOD)
if findings:
    for f in findings:
        print_finding(f)
else:
    print("  ✅ Уязвимостей не найдено — аудитор одобрил функцию.")


## 7. Та же атака на безопасную функцию

Передаём тот же payload `x' OR '1'='1`. Сравним поведение.


In [ ]:
section("АТАКА на безопасную функцию")
payload = "x' OR '1'='1"
print(f"  Атакующий снова прислал login = {payload!r}")
rows = get_user_GOOD(conn, payload)
print(f"\n  ✅ Получено {len(rows)} строк (драйвер искал пользователя с буквально таким логином):")
show_result(rows)


section("Атака с UNION тоже мимо")
payload2 = "x' UNION SELECT id, login, password_hash, role FROM users --"
rows = get_user_GOOD(conn, payload2)
print(f"  ✅ Получено {len(rows)} строк — атакующий ушёл с пустыми руками.")


## Итог

Мы увидели одно и то же на двух функциях:

- **Уязвимая** — украли данные / повредили БД / поднялись в правах.
- **Безопасная** — та же атака уходит в пустоту.

Между ними — **один аудитор** с конкретным правилом, которое можно
запустить детерминированно (без LLM) на каждом сгенерированном SQL.

## Куда дальше

- **Описание уязвимости (под микроскопом):** [problems/vulnerabilities/01-sql-injection-classic/README.md](../problems/vulnerabilities/01-sql-injection-classic/README.md)
- **Варианты решения + почему так:** [problems/vulnerabilities/01-sql-injection-classic/solutions.md](../problems/vulnerabilities/01-sql-injection-classic/solutions.md)
- **Архитектура цикла:** [docs/adr/0002-loop-architecture-langgraph.md](../docs/adr/0002-loop-architecture-langgraph.md)
- **Гибридный аудитор (pglast + LLM):** [docs/adr/0004-hybrid-auditor-ast-plus-llm.md](../docs/adr/0004-hybrid-auditor-ast-plus-llm.md)
